# AgentBench-Local: Kaggle GPU Benchmark Runner

This notebook benchmarks large local models (7B, 9B, 14B) on a Kaggle GPU (T4 / P100) running the AgentBench-Local test suite.

### Prerequisites on Kaggle:
1. **Accelerator:** GPU T4 x2 or P100
2. **Internet:** Enabled in notebook settings
3. **Persistence:** Files saved in `/kaggle/working/runs`

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

In [ ]:
# 2. Install and launch Ollama
import subprocess, time, urllib.request, json

print("Installing Ollama...")
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

print("Starting Ollama daemon in background...")
subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)

# Wait for daemon to respond
for i in range(30):
    try:
        with urllib.request.urlopen("http://localhost:11434/api/version", timeout=2) as resp:
            data = json.loads(resp.read().decode())
            print(f"Ollama daemon active! Version: {data.get('version')}")
            break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama failed to start in 30 seconds")

In [ ]:
# 3. Pull heavy models for benchmark panel
!ollama pull qwen2.5:7b
!ollama pull deepseek-r1:14b
!ollama list

In [ ]:
# 4. Install repository dependencies
!pip install -q openai pydantic pyyaml pandas matplotlib pytest tqdm

In [ ]:
# 5. Run the full evaluation pipeline for heavy models on frozen test split (3 repeats)
# Output will be saved directly into /kaggle/working/runs
!python -m src.run \
    --models qwen2.5:7b deepseek-r1:14b \
    --configs A_baseline B_validate_retry C_guards \
    --split test \
    --repeats 3 \
    --runs-dir /kaggle/working/runs

In [ ]:
# 6. Generate summary reports and charts
!python -m src.report \
    --runs-dir /kaggle/working/runs \
    --out-dir /kaggle/working/results

# Zip output runs for easy download
!zip -r /kaggle/working/runs_kaggle.zip /kaggle/working/runs